In [5]:
#Simulated Annealing
import random
import math

def calculate_conflicts(board):
    """Calculates the number of conflicting pairs of queens on the board."""
    conflicts = 0
    n = len(board)
    for i in range(n):
        for j in range(i + 1, n):
            if board[i] == board[j] or abs(board[i] - board[j]) == abs(i - j):
                conflicts += 1
    return conflicts

def get_neighbor(board):
    """Generates a neighbor state by shifting one queen to a different row."""
    neighbor = list(board)
    col = random.randint(0, 7)
    current_row = board[col]

    new_row = random.randint(0, 7)
    while new_row == current_row:
        new_row = random.randint(0, 7)

    neighbor[col] = new_row
    return neighbor

def simulated_annealing():
    """Solves 8-Queens while explicitly tracking the absolute best solution found."""
    # 1. Initialize parameters
    temperature = 100.0
    cooling_rate = 0.995  # Slightly slower cooling for better exploration
    min_temperature = 0.001
    max_iterations = 50000

    # 2. Setup current and global best trackers
    current_state = [random.randint(0, 7) for _ in range(8)]
    current_cost = calculate_conflicts(current_state)

    best_state = list(current_state)
    best_cost = current_cost

    iteration = 0

    # 3. Optimization loop
    while temperature > min_temperature and best_cost > 0 and iteration < max_iterations:
        neighbor_state = get_neighbor(current_state)
        neighbor_cost = calculate_conflicts(neighbor_state)

        delta_e = neighbor_cost - current_cost

        # Decide whether to accept the new state
        if delta_e < 0:
            current_state = neighbor_state
            current_cost = neighbor_cost
        else:
            acceptance_probability = math.exp(-delta_e / temperature)
            if random.random() < acceptance_probability:
                current_state = neighbor_state
                current_cost = neighbor_cost

        # KEY ADDITION: Track the absolute lowest minimum value and best solution seen so far
        if current_cost < best_cost:
            best_state = list(current_state)
            best_cost = current_cost

        temperature *= cooling_rate
        iteration += 1

    return best_state, best_cost, iteration

def print_board(board):
    """Prints a visual 8x8 chessboard representation."""
    for row in range(8):
        line = ""
        for col in range(8):
            if board[col] == row:
                line += " Q "
            else:
                line += " . "
        print(line)

# Run and output the absolute minimum metrics
if __name__ == "__main__":
    best_sol, min_conflicts, total_iters = simulated_annealing()

    print("\n--- Execution Finished ---")
    print(f"Total Iterations Tracked: {total_iters}")
    print(f"Minimum Value (Lowest Conflicts Discovered): {min_conflicts}")
    print(f"Best Solution Found: {best_sol}\n")

    if min_conflicts == 0:
        print("Success! Perfect solution found:")
    else:
        print(f"Local optimum reached. Best possible layout with {min_conflicts} conflicts:")

    print_board(best_sol)



--- Execution Finished ---
Total Iterations Tracked: 1495
Minimum Value (Lowest Conflicts Discovered): 0
Best Solution Found: [5, 1, 6, 0, 3, 7, 4, 2]

Success! Perfect solution found:
 .  .  .  Q  .  .  .  . 
 .  Q  .  .  .  .  .  . 
 .  .  .  .  .  .  .  Q 
 .  .  .  .  Q  .  .  . 
 .  .  .  .  .  .  Q  . 
 Q  .  .  .  .  .  .  . 
 .  .  Q  .  .  .  .  . 
 .  .  .  .  .  Q  .  . 


In [13]:
#Propositional Logic

import itertools

class Sentence:
    def evaluate(self, model): raise NotImplementedError

class Symbol(Sentence):
    def __init__(self, name):
        self.name = name
    def evaluate(self, model):
        return model[self.name]

class Not(Sentence):
    def __init__(self, sub):
        self.sub = sub
    def evaluate(self, model):
        return not self.sub.evaluate(model)

class And(Sentence):
    def __init__(self, left, right):
        self.left = left
        self.right = right
    def evaluate(self, model):
        return self.left.evaluate(model) and self.right.evaluate(model)

class Or(Sentence):
    def __init__(self, left, right):
        self.left = left
        self.right = right
    def evaluate(self, model):
        return self.left.evaluate(model) or self.right.evaluate(model)

def format_bool(val):
    return "true" if val else "false"

def generate_truth_table():
    # 1. Define Symbols
    A = Symbol('A')
    B = Symbol('B')
    C = Symbol('C')

    # 2. Define the Intermediate Clauses, KB, and Alpha from the slide
    clause1 = Or(A, C)           # A v C
    clause2 = Or(B, Not(C))      # B v ~C
    kb = And(clause1, clause2)   # KB = (A v C) ^ (B v ~C)
    alpha = Or(A, B)             # alpha = A v B

    # 3. Print the Header (matching the columns in the slide image)
    header = f"{'A':<8} | {'B':<8} | {'C':<8} | {'A v C':<8} | {'B v ~C':<8} | {'KB':<8} | {'alpha':<8}"
    print(header)
    print("-" * len(header))

    # All symbols to iterate over
    symbols = ['A', 'B', 'C']
    entails = True

    # 4. Enumerate all truth assignments (False first to match slide order)
    for combination in itertools.product([False, True], repeat=3):
        model = dict(zip(symbols, combination))

        # Evaluate expressions
        v_A = model['A']
        v_B = model['B']
        v_C = model['C']
        v_c1 = clause1.evaluate(model)
        v_c2 = clause2.evaluate(model)
        v_kb = kb.evaluate(model)
        v_alpha = alpha.evaluate(model)

        # Check entailment condition: if KB is true, alpha must be true
        if v_kb and not v_alpha:
            entails = False

        # Print row
        print(f"{format_bool(v_A):<8} | {format_bool(v_B):<8} | {format_bool(v_C):<8} | "
              f"{format_bool(v_c1):<8} | {format_bool(v_c2):<8} | {format_bool(v_kb):<8} | {format_bool(v_alpha):<8}")

    print("-" * len(header))
    # 5. Output conclusion
    if entails:
        print("\nConclusion: KB ⊨ α (The Knowledge Base entails Alpha).")
    else:
        print("\nConclusion: KB does NOT entail Alpha.")

if __name__ == "__main__":
    generate_truth_table()


A        | B        | C        | A v C    | B v ~C   | KB       | alpha   
--------------------------------------------------------------------------
false    | false    | false    | false    | true     | false    | false   
false    | false    | true     | true     | false    | false    | false   
false    | true     | false    | false    | true     | false    | true    
false    | true     | true     | true     | true     | true     | true    
true     | false    | false    | true     | true     | true     | true    
true     | false    | true     | true     | false    | false    | true    
true     | true     | false    | true     | true     | true     | true    
true     | true     | true     | true     | true     | true     | true    
--------------------------------------------------------------------------

Conclusion: KB ⊨ α (The Knowledge Base entails Alpha).
